# Linear Discriminant Analysis (LDA) - From Scratch Implementation

## Table of Contents
1. [Theory & Mathematical Foundation](#theory)
2. [Implementation from Scratch](#implementation)
3. [Training & Optimization](#training)
4. [Diagnostics & Evaluation](#diagnostics)
5. [Visualizations](#visualizations)
6. [Use Cases & Guidelines](#use-cases)
7. [Comparison with sklearn](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as SklearnLDA
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Theory & Mathematical Foundation <a id='theory'></a>

### What is Linear Discriminant Analysis?

Linear Discriminant Analysis (LDA) is a **supervised dimensionality reduction** technique that projects data onto a lower-dimensional space while **maximizing class separability**. Unlike PCA which finds directions of maximum variance, LDA finds directions that maximize the separation between multiple classes.

### Fisher's Criterion

LDA is based on **Fisher's Linear Discriminant**, which seeks to find a projection that maximizes the ratio of between-class variance to within-class variance.

#### For two classes, Fisher's criterion is:
$$J(w) = \frac{w^T S_B w}{w^T S_W w}$$

Where:
- $w$ is the projection vector
- $S_B$ is the between-class scatter matrix
- $S_W$ is the within-class scatter matrix

### Scatter Matrices

#### Within-Class Scatter Matrix ($S_W$):
Measures how spread out the data is within each class.

$$S_W = \sum_{c=1}^{C} \sum_{x_i \in c} (x_i - \mu_c)(x_i - \mu_c)^T$$

Where $\mu_c$ is the mean of class $c$.

#### Between-Class Scatter Matrix ($S_B$):
Measures how spread out the class means are from the overall mean.

$$S_B = \sum_{c=1}^{C} n_c (\mu_c - \mu)(\mu_c - \mu)^T$$

Where $\mu$ is the overall mean and $n_c$ is the number of samples in class $c$.

### Solution via Eigendecomposition

The optimal projection directions are found by solving the **generalized eigenvalue problem**:

$$S_B w = \lambda S_W w$$

Or equivalently (when $S_W$ is invertible):

$$S_W^{-1} S_B w = \lambda w$$

The eigenvectors corresponding to the largest eigenvalues give the directions of maximum class separation.

### Maximum Number of Components

**Important constraint**: LDA can produce at most **min(n_features, n_classes - 1)** discriminant components.

This is because:
- $S_B$ has rank at most $C - 1$ (where $C$ is the number of classes)
- The between-class scatter matrix is formed from $C$ class means, but they lie in a $(C-1)$-dimensional subspace

### LDA vs PCA Comparison

| Aspect | PCA | LDA |
|--------|-----|-----|
| **Type** | Unsupervised | Supervised |
| **Objective** | Maximize variance | Maximize class separability |
| **Uses labels** | No | Yes |
| **Max components** | min(n_samples, n_features) | min(n_features, n_classes - 1) |
| **Best for** | General dimensionality reduction | Classification preprocessing |
| **Assumptions** | None about class distribution | Normal distribution, equal covariance |

### Time Complexity
- Computing scatter matrices: $O(n \cdot d^2)$ where $n$ = samples, $d$ = features
- Eigendecomposition: $O(d^3)$
- Transform: $O(n \cdot d \cdot k)$ where $k$ = components

In [ ]:
# Visual demonstration of Fisher's criterion
def visualize_fisher_criterion():
    """
    Visualize the concept of within-class and between-class scatter.
    """
    # Generate two-class data
    np.random.seed(42)
    
    # Class 1: centered at (2, 2)
    class1 = np.random.randn(50, 2) * 0.8 + [2, 2]
    # Class 2: centered at (5, 4)
    class2 = np.random.randn(50, 2) * 0.8 + [5, 4]
    
    # Compute means
    mean1 = np.mean(class1, axis=0)
    mean2 = np.mean(class2, axis=0)
    overall_mean = np.mean(np.vstack([class1, class2]), axis=0)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Plot 1: Original data with class means
    axes[0].scatter(class1[:, 0], class1[:, 1], c='blue', alpha=0.6, label='Class 1', s=50)
    axes[0].scatter(class2[:, 0], class2[:, 1], c='red', alpha=0.6, label='Class 2', s=50)
    axes[0].scatter(*mean1, c='blue', marker='X', s=200, edgecolor='black', linewidth=2, label='Mean 1')
    axes[0].scatter(*mean2, c='red', marker='X', s=200, edgecolor='black', linewidth=2, label='Mean 2')
    axes[0].scatter(*overall_mean, c='green', marker='D', s=200, edgecolor='black', linewidth=2, label='Overall Mean')
    axes[0].set_title('Original Data with Class Means', fontsize=12)
    axes[0].legend()
    axes[0].set_xlabel('Feature 1')
    axes[0].set_ylabel('Feature 2')
    
    # Plot 2: Within-class scatter visualization
    axes[1].scatter(class1[:, 0], class1[:, 1], c='blue', alpha=0.6, s=50)
    axes[1].scatter(class2[:, 0], class2[:, 1], c='red', alpha=0.6, s=50)
    # Draw lines from points to their class mean
    for point in class1[::5]:  # Sample every 5th point
        axes[1].plot([point[0], mean1[0]], [point[1], mean1[1]], 'b-', alpha=0.3)
    for point in class2[::5]:
        axes[1].plot([point[0], mean2[0]], [point[1], mean2[1]], 'r-', alpha=0.3)
    axes[1].scatter(*mean1, c='blue', marker='X', s=200, edgecolor='black', linewidth=2)
    axes[1].scatter(*mean2, c='red', marker='X', s=200, edgecolor='black', linewidth=2)
    axes[1].set_title('Within-Class Scatter (minimize)', fontsize=12)
    axes[1].set_xlabel('Feature 1')
    axes[1].set_ylabel('Feature 2')
    
    # Plot 3: Between-class scatter visualization
    axes[2].scatter(class1[:, 0], class1[:, 1], c='blue', alpha=0.3, s=50)
    axes[2].scatter(class2[:, 0], class2[:, 1], c='red', alpha=0.3, s=50)
    axes[2].scatter(*mean1, c='blue', marker='X', s=200, edgecolor='black', linewidth=2, label='Mean 1')
    axes[2].scatter(*mean2, c='red', marker='X', s=200, edgecolor='black', linewidth=2, label='Mean 2')
    axes[2].scatter(*overall_mean, c='green', marker='D', s=200, edgecolor='black', linewidth=2, label='Overall Mean')
    # Draw lines from class means to overall mean
    axes[2].annotate('', xy=mean1, xytext=overall_mean,
                     arrowprops=dict(arrowstyle='->', color='blue', lw=3))
    axes[2].annotate('', xy=mean2, xytext=overall_mean,
                     arrowprops=dict(arrowstyle='->', color='red', lw=3))
    axes[2].set_title('Between-Class Scatter (maximize)', fontsize=12)
    axes[2].set_xlabel('Feature 1')
    axes[2].set_ylabel('Feature 2')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()

visualize_fisher_criterion()

## 2. Implementation from Scratch <a id='implementation'></a>

In [ ]:
class LDAScratch:
    """
    Linear Discriminant Analysis implementation from scratch.
    
    LDA finds the linear combinations of features that best separate
    two or more classes of objects or events.
    
    Parameters:
    -----------
    n_components : int, default=None
        Number of components for dimensionality reduction.
        If None, will be set to min(n_features, n_classes - 1).
    
    Attributes:
    -----------
    components_ : ndarray of shape (n_components, n_features)
        The linear discriminant directions (eigenvectors).
    explained_variance_ratio_ : ndarray of shape (n_components,)
        Percentage of variance explained by each component.
    means_ : ndarray of shape (n_classes, n_features)
        Class means.
    classes_ : ndarray of shape (n_classes,)
        Unique class labels.
    eigenvalues_ : ndarray of shape (n_components,)
        Eigenvalues corresponding to each component.
    """
    
    def __init__(self, n_components=None):
        self.n_components = n_components
        self.components_ = None
        self.explained_variance_ratio_ = None
        self.means_ = None
        self.classes_ = None
        self.eigenvalues_ = None
        self._overall_mean = None
        self._S_W = None  # Within-class scatter matrix
        self._S_B = None  # Between-class scatter matrix
    
    def _compute_scatter_matrices(self, X, y):
        """
        Compute within-class and between-class scatter matrices.
        
        Parameters:
        -----------
        X : ndarray of shape (n_samples, n_features)
            Training data.
        y : ndarray of shape (n_samples,)
            Target values.
        
        Returns:
        --------
        S_W : ndarray of shape (n_features, n_features)
            Within-class scatter matrix.
        S_B : ndarray of shape (n_features, n_features)
            Between-class scatter matrix.
        """
        n_features = X.shape[1]
        
        # Initialize scatter matrices
        S_W = np.zeros((n_features, n_features))
        S_B = np.zeros((n_features, n_features))
        
        # Compute overall mean
        self._overall_mean = np.mean(X, axis=0)
        
        # Compute scatter matrices for each class
        for c in self.classes_:
            # Get samples belonging to class c
            X_c = X[y == c]
            n_c = X_c.shape[0]
            
            # Compute class mean
            mean_c = np.mean(X_c, axis=0)
            
            # Within-class scatter: sum of (x - mean_c)(x - mean_c)^T
            # Efficient computation using matrix operations
            X_centered = X_c - mean_c
            S_W += X_centered.T @ X_centered
            
            # Between-class scatter: n_c * (mean_c - overall_mean)(mean_c - overall_mean)^T
            mean_diff = (mean_c - self._overall_mean).reshape(-1, 1)
            S_B += n_c * (mean_diff @ mean_diff.T)
        
        return S_W, S_B
    
    def fit(self, X, y):
        """
        Fit the LDA model.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training data.
        y : array-like of shape (n_samples,)
            Target values.
        
        Returns:
        --------
        self : object
            Fitted estimator.
        """
        # Convert to numpy arrays
        X = np.array(X, dtype=np.float64)
        y = np.array(y)
        
        n_samples, n_features = X.shape
        
        # Get unique classes and compute class means
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        
        # Validate n_components
        max_components = min(n_features, n_classes - 1)
        if self.n_components is None:
            self.n_components = max_components
        elif self.n_components > max_components:
            print(f"Warning: n_components={self.n_components} is greater than "
                  f"max possible ({max_components}). Setting to {max_components}.")
            self.n_components = max_components
        
        # Compute class means
        self.means_ = np.array([X[y == c].mean(axis=0) for c in self.classes_])
        
        # Compute scatter matrices
        self._S_W, self._S_B = self._compute_scatter_matrices(X, y)
        
        # Solve the generalized eigenvalue problem: S_B * w = lambda * S_W * w
        # Equivalent to: S_W^{-1} * S_B * w = lambda * w
        # For numerical stability, we use a small regularization
        S_W_reg = self._S_W + np.eye(n_features) * 1e-6
        
        # Compute S_W^{-1} * S_B
        try:
            S_W_inv = np.linalg.inv(S_W_reg)
        except np.linalg.LinAlgError:
            # If still singular, use pseudo-inverse
            S_W_inv = np.linalg.pinv(S_W_reg)
        
        A = S_W_inv @ self._S_B
        
        # Compute eigenvalues and eigenvectors
        eigenvalues, eigenvectors = np.linalg.eig(A)
        
        # Eigenvalues might be complex due to numerical issues
        # Take only the real parts
        eigenvalues = np.real(eigenvalues)
        eigenvectors = np.real(eigenvectors)
        
        # Sort eigenvectors by eigenvalues in descending order
        sorted_indices = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[sorted_indices]
        eigenvectors = eigenvectors[:, sorted_indices]
        
        # Select top n_components
        self.eigenvalues_ = eigenvalues[:self.n_components]
        self.components_ = eigenvectors[:, :self.n_components].T
        
        # Compute explained variance ratio
        total_eigenvalue = np.sum(np.abs(eigenvalues[eigenvalues > 1e-10]))
        if total_eigenvalue > 0:
            self.explained_variance_ratio_ = np.abs(self.eigenvalues_) / total_eigenvalue
        else:
            self.explained_variance_ratio_ = np.zeros(self.n_components)
        
        return self
    
    def transform(self, X):
        """
        Project data onto the LDA components.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Data to transform.
        
        Returns:
        --------
        X_new : ndarray of shape (n_samples, n_components)
            Transformed data.
        """
        X = np.array(X, dtype=np.float64)
        return X @ self.components_.T
    
    def fit_transform(self, X, y):
        """
        Fit the model and transform the data.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training data.
        y : array-like of shape (n_samples,)
            Target values.
        
        Returns:
        --------
        X_new : ndarray of shape (n_samples, n_components)
            Transformed data.
        """
        self.fit(X, y)
        return self.transform(X)
    
    def get_params(self):
        """
        Get model parameters.
        
        Returns:
        --------
        params : dict
            Dictionary of model parameters.
        """
        return {
            'n_components': self.n_components,
            'n_classes': len(self.classes_) if self.classes_ is not None else None,
            'explained_variance_ratio': self.explained_variance_ratio_
        }

In [ ]:
# Quick test of our implementation
print("Testing LDA implementation...")
print("=" * 50)

# Generate simple test data
np.random.seed(42)
X_test = np.vstack([
    np.random.randn(30, 4) + [2, 2, 2, 2],
    np.random.randn(30, 4) + [5, 5, 5, 5],
    np.random.randn(30, 4) + [8, 2, 8, 2]
])
y_test = np.array([0]*30 + [1]*30 + [2]*30)

# Fit our LDA
lda = LDAScratch(n_components=2)
X_transformed = lda.fit_transform(X_test, y_test)

print(f"Original shape: {X_test.shape}")
print(f"Transformed shape: {X_transformed.shape}")
print(f"Number of classes: {len(lda.classes_)}")
print(f"Max possible components: {len(lda.classes_) - 1}")
print(f"Components used: {lda.n_components}")
print(f"Explained variance ratio: {lda.explained_variance_ratio_}")
print(f"\nComponents shape: {lda.components_.shape}")

## 3. Training & Optimization <a id='training'></a>

In [ ]:
# Load the Wine dataset
wine = load_wine()
X = wine.data
y = wine.target
feature_names = wine.feature_names
class_names = wine.target_names

print("Wine Dataset Information")
print("=" * 50)
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Class names: {class_names}")
print(f"Class distribution: {np.bincount(y)}")
print(f"\nFeature names:")
for i, name in enumerate(feature_names):
    print(f"  {i+1}. {name}")

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features (important for LDA)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train_scaled.shape}")
print(f"Test set size: {X_test_scaled.shape}")
print(f"Class distribution in training: {np.bincount(y_train)}")
print(f"Class distribution in test: {np.bincount(y_test)}")

In [ ]:
# Train LDA with different numbers of components
print("Training LDA with different components")
print("=" * 50)

n_classes = len(np.unique(y_train))
max_components = n_classes - 1  # For Wine dataset: 3-1 = 2

print(f"Number of classes: {n_classes}")
print(f"Maximum possible LDA components: {max_components}")
print()

# Train with all possible components
for n_comp in range(1, max_components + 1):
    lda = LDAScratch(n_components=n_comp)
    X_train_lda = lda.fit_transform(X_train_scaled, y_train)
    
    print(f"n_components = {n_comp}:")
    print(f"  Transformed shape: {X_train_lda.shape}")
    print(f"  Explained variance ratio: {lda.explained_variance_ratio_}")
    print(f"  Cumulative variance: {np.sum(lda.explained_variance_ratio_):.4f}")
    print()

In [ ]:
# Final model with 2 components (maximum for 3 classes)
lda_final = LDAScratch(n_components=2)
X_train_lda = lda_final.fit_transform(X_train_scaled, y_train)
X_test_lda = lda_final.transform(X_test_scaled)

print("Final LDA Model")
print("=" * 50)
print(f"Components: {lda_final.n_components}")
print(f"Eigenvalues: {lda_final.eigenvalues_}")
print(f"Explained variance ratio: {lda_final.explained_variance_ratio_}")
print(f"Total explained variance: {np.sum(lda_final.explained_variance_ratio_):.4f}")
print(f"\nTransformed training shape: {X_train_lda.shape}")
print(f"Transformed test shape: {X_test_lda.shape}")

## 4. Diagnostics & Evaluation <a id='diagnostics'></a>

In [ ]:
# Explained variance analysis
def plot_explained_variance(lda_model, title="LDA Explained Variance"):
    """
    Plot explained variance ratio for LDA components.
    """
    n_components = len(lda_model.explained_variance_ratio_)
    cumulative_variance = np.cumsum(lda_model.explained_variance_ratio_)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Bar plot for individual variance
    x = np.arange(1, n_components + 1)
    bars = ax.bar(x, lda_model.explained_variance_ratio_, alpha=0.7, 
                  color='steelblue', label='Individual')
    
    # Line plot for cumulative variance
    ax.plot(x, cumulative_variance, 'ro-', linewidth=2, 
            markersize=8, label='Cumulative')
    
    # Add value labels on bars
    for bar, val in zip(bars, lda_model.explained_variance_ratio_):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2%}', ha='center', va='bottom', fontsize=10)
    
    ax.set_xlabel('LDA Component', fontsize=12)
    ax.set_ylabel('Explained Variance Ratio', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels([f'LD{i}' for i in x])
    ax.legend(loc='center right')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return cumulative_variance

cumulative = plot_explained_variance(lda_final, "LDA Explained Variance - Wine Dataset")

In [ ]:
# Classification accuracy after projection
def evaluate_classification_accuracy(X_train, X_test, y_train, y_test, title=""):
    """
    Evaluate classification accuracy using different classifiers.
    """
    classifiers = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'K-Nearest Neighbors (k=5)': KNeighborsClassifier(n_neighbors=5),
        'K-Nearest Neighbors (k=3)': KNeighborsClassifier(n_neighbors=3)
    }
    
    results = {}
    
    print(f"Classification Results {title}")
    print("=" * 60)
    
    for name, clf in classifiers.items():
        clf.fit(X_train, y_train)
        train_acc = clf.score(X_train, y_train)
        test_acc = clf.score(X_test, y_test)
        results[name] = {'train': train_acc, 'test': test_acc}
        
        print(f"\n{name}:")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Test Accuracy:  {test_acc:.4f}")
    
    return results

# Compare original features vs LDA-transformed features
print("\n" + "="*60)
print("USING ORIGINAL FEATURES (13 dimensions)")
print("="*60)
results_original = evaluate_classification_accuracy(
    X_train_scaled, X_test_scaled, y_train, y_test, "(Original 13 features)"
)

print("\n" + "="*60)
print("USING LDA-TRANSFORMED FEATURES (2 dimensions)")
print("="*60)
results_lda = evaluate_classification_accuracy(
    X_train_lda, X_test_lda, y_train, y_test, "(LDA 2 components)"
)

In [ ]:
# Visualization of accuracy comparison
def plot_accuracy_comparison(results_original, results_lda):
    """
    Compare classification accuracy between original and LDA features.
    """
    classifiers = list(results_original.keys())
    x = np.arange(len(classifiers))
    width = 0.35
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Training accuracy
    train_orig = [results_original[c]['train'] for c in classifiers]
    train_lda = [results_lda[c]['train'] for c in classifiers]
    
    axes[0].bar(x - width/2, train_orig, width, label='Original (13 features)', color='steelblue')
    axes[0].bar(x + width/2, train_lda, width, label='LDA (2 components)', color='coral')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Training Accuracy Comparison')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(['LR', 'KNN-5', 'KNN-3'])
    axes[0].legend()
    axes[0].set_ylim(0.8, 1.02)
    axes[0].grid(axis='y', alpha=0.3)
    
    # Test accuracy
    test_orig = [results_original[c]['test'] for c in classifiers]
    test_lda = [results_lda[c]['test'] for c in classifiers]
    
    axes[1].bar(x - width/2, test_orig, width, label='Original (13 features)', color='steelblue')
    axes[1].bar(x + width/2, test_lda, width, label='LDA (2 components)', color='coral')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Test Accuracy Comparison')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(['LR', 'KNN-5', 'KNN-3'])
    axes[1].legend()
    axes[1].set_ylim(0.8, 1.02)
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.suptitle('Dimensionality Reduction: 13 features -> 2 LDA components', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

plot_accuracy_comparison(results_original, results_lda)

In [ ]:
# Detailed classification report for best classifier on LDA features
print("Detailed Classification Report (Logistic Regression on LDA features)")
print("=" * 70)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_lda, y_train)
y_pred = clf.predict(X_test_lda)

print(classification_report(y_test, y_pred, target_names=class_names))

## 5. Visualizations <a id='visualizations'></a>

In [ ]:
# 2D projection colored by class
def plot_lda_projection(X_lda, y, class_names, title="LDA Projection"):
    """
    Plot 2D LDA projection with class labels.
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    markers = ['o', 's', '^']
    
    for i, (color, marker, name) in enumerate(zip(colors, markers, class_names)):
        mask = y == i
        ax.scatter(X_lda[mask, 0], X_lda[mask, 1], 
                   c=color, marker=marker, s=80, alpha=0.7,
                   label=name, edgecolors='black', linewidth=0.5)
    
    ax.set_xlabel('LD1 (Linear Discriminant 1)', fontsize=12)
    ax.set_ylabel('LD2 (Linear Discriminant 2)', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend(title='Wine Class', fontsize=10)
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Plot training data projection
plot_lda_projection(X_train_lda, y_train, class_names, 
                    "LDA Projection - Wine Dataset (Training Data)")

In [ ]:
# Comparison with PCA projection
def compare_lda_pca(X_train, X_test, y_train, y_test, class_names):
    """
    Compare LDA and PCA projections side by side.
    """
    # Fit both models
    lda = LDAScratch(n_components=2)
    pca = PCA(n_components=2)
    
    X_lda = lda.fit_transform(X_train, y_train)
    X_pca = pca.fit_transform(X_train)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    markers = ['o', 's', '^']
    
    # LDA plot
    for i, (color, marker, name) in enumerate(zip(colors, markers, class_names)):
        mask = y_train == i
        axes[0].scatter(X_lda[mask, 0], X_lda[mask, 1], 
                        c=color, marker=marker, s=80, alpha=0.7,
                        label=name, edgecolors='black', linewidth=0.5)
    
    axes[0].set_xlabel('LD1', fontsize=12)
    axes[0].set_ylabel('LD2', fontsize=12)
    axes[0].set_title(f'LDA Projection\n(Explained Variance: {np.sum(lda.explained_variance_ratio_):.2%})', 
                      fontsize=14)
    axes[0].legend(title='Class')
    axes[0].grid(alpha=0.3)
    
    # PCA plot
    for i, (color, marker, name) in enumerate(zip(colors, markers, class_names)):
        mask = y_train == i
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                        c=color, marker=marker, s=80, alpha=0.7,
                        label=name, edgecolors='black', linewidth=0.5)
    
    axes[1].set_xlabel('PC1', fontsize=12)
    axes[1].set_ylabel('PC2', fontsize=12)
    axes[1].set_title(f'PCA Projection\n(Explained Variance: {np.sum(pca.explained_variance_ratio_):.2%})', 
                      fontsize=14)
    axes[1].legend(title='Class')
    axes[1].grid(alpha=0.3)
    
    plt.suptitle('LDA vs PCA: Supervised vs Unsupervised Dimensionality Reduction', 
                 fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()
    
    return lda, pca

lda_model, pca_model = compare_lda_pca(X_train_scaled, X_test_scaled, 
                                        y_train, y_test, class_names)

In [ ]:
# Class separation visualization
def visualize_class_separation(X_lda, y, class_names):
    """
    Visualize class separation in LDA space using multiple methods.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    
    # Plot 1: Distribution along LD1
    for i, (color, name) in enumerate(zip(colors, class_names)):
        mask = y == i
        axes[0, 0].hist(X_lda[mask, 0], bins=15, alpha=0.6, color=color, 
                        label=name, edgecolor='black')
    axes[0, 0].set_xlabel('LD1', fontsize=12)
    axes[0, 0].set_ylabel('Frequency', fontsize=12)
    axes[0, 0].set_title('Class Distribution along LD1', fontsize=14)
    axes[0, 0].legend(title='Class')
    axes[0, 0].grid(alpha=0.3)
    
    # Plot 2: Distribution along LD2
    for i, (color, name) in enumerate(zip(colors, class_names)):
        mask = y == i
        axes[0, 1].hist(X_lda[mask, 1], bins=15, alpha=0.6, color=color, 
                        label=name, edgecolor='black')
    axes[0, 1].set_xlabel('LD2', fontsize=12)
    axes[0, 1].set_ylabel('Frequency', fontsize=12)
    axes[0, 1].set_title('Class Distribution along LD2', fontsize=14)
    axes[0, 1].legend(title='Class')
    axes[0, 1].grid(alpha=0.3)
    
    # Plot 3: Box plots for each component
    data_ld1 = [X_lda[y == i, 0] for i in range(len(class_names))]
    bp1 = axes[1, 0].boxplot(data_ld1, labels=class_names, patch_artist=True)
    for patch, color in zip(bp1['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    axes[1, 0].set_xlabel('Class', fontsize=12)
    axes[1, 0].set_ylabel('LD1 Value', fontsize=12)
    axes[1, 0].set_title('Class Separation on LD1', fontsize=14)
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    # Plot 4: KDE plot for 2D density
    for i, (color, name) in enumerate(zip(colors, class_names)):
        mask = y == i
        sns.kdeplot(x=X_lda[mask, 0], y=X_lda[mask, 1], 
                    ax=axes[1, 1], color=color, label=name,
                    levels=5, linewidths=1.5)
        axes[1, 1].scatter(X_lda[mask, 0], X_lda[mask, 1], 
                           c=color, alpha=0.3, s=30)
    axes[1, 1].set_xlabel('LD1', fontsize=12)
    axes[1, 1].set_ylabel('LD2', fontsize=12)
    axes[1, 1].set_title('2D Density Contours', fontsize=14)
    axes[1, 1].legend(title='Class')
    axes[1, 1].grid(alpha=0.3)
    
    plt.suptitle('Class Separation Analysis in LDA Space', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

visualize_class_separation(X_train_lda, y_train, class_names)

In [ ]:
# Decision boundary visualization
def plot_decision_boundary_lda(X_lda, y, class_names):
    """
    Plot decision boundaries using a classifier in LDA space.
    """
    # Train a classifier
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_lda, y)
    
    # Create mesh grid
    h = 0.05
    x_min, x_max = X_lda[:, 0].min() - 1, X_lda[:, 0].max() + 1
    y_min, y_max = X_lda[:, 1].min() - 1, X_lda[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict on mesh
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Plot decision regions
    cmap_light = plt.cm.RdYlBu
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_light)
    ax.contour(xx, yy, Z, colors='black', linewidths=0.5, alpha=0.5)
    
    # Plot data points
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    markers = ['o', 's', '^']
    
    for i, (color, marker, name) in enumerate(zip(colors, markers, class_names)):
        mask = y == i
        ax.scatter(X_lda[mask, 0], X_lda[mask, 1], 
                   c=color, marker=marker, s=100, alpha=0.8,
                   label=name, edgecolors='black', linewidth=1)
    
    ax.set_xlabel('LD1', fontsize=12)
    ax.set_ylabel('LD2', fontsize=12)
    ax.set_title('Decision Boundaries in LDA Space\n(Logistic Regression Classifier)', fontsize=14)
    ax.legend(title='Wine Class', loc='best')
    
    plt.tight_layout()
    plt.show()

plot_decision_boundary_lda(X_train_lda, y_train, class_names)

In [ ]:
# Feature contribution to LDA components
def plot_feature_contributions(lda_model, feature_names, n_top=10):
    """
    Visualize which original features contribute most to LDA components.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    for i in range(min(2, lda_model.n_components)):
        # Get component weights
        weights = lda_model.components_[i]
        
        # Sort by absolute value
        sorted_idx = np.argsort(np.abs(weights))[::-1][:n_top]
        top_features = [feature_names[j] for j in sorted_idx]
        top_weights = weights[sorted_idx]
        
        # Create color based on sign
        colors = ['steelblue' if w >= 0 else 'coral' for w in top_weights]
        
        # Horizontal bar plot
        y_pos = np.arange(len(top_features))
        axes[i].barh(y_pos, top_weights, color=colors, alpha=0.7, edgecolor='black')
        axes[i].set_yticks(y_pos)
        axes[i].set_yticklabels(top_features)
        axes[i].axvline(x=0, color='black', linewidth=0.5)
        axes[i].set_xlabel('Weight', fontsize=12)
        axes[i].set_title(f'LD{i+1} Feature Contributions', fontsize=14)
        axes[i].grid(axis='x', alpha=0.3)
    
    plt.suptitle('Feature Contributions to LDA Components', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

plot_feature_contributions(lda_final, feature_names)

## 6. Use Cases & Guidelines <a id='use-cases'></a>

### When to Use LDA

#### Good Use Cases:

1. **Classification Preprocessing**
   - Reduce dimensionality while preserving class separability
   - Improve classifier performance by removing noise
   - Speed up training of downstream classifiers

2. **Class Separation & Visualization**
   - Visualize high-dimensional data in 2D/3D
   - Understand class structure in the data
   - Identify which features separate classes best

3. **Feature Extraction**
   - Create discriminative features from raw data
   - Face recognition and biometrics
   - Medical diagnosis (disease vs. healthy)

4. **Multiclass Problems**
   - When you have multiple well-defined classes
   - Reducing to n_classes - 1 dimensions is sufficient

### When NOT to Use LDA

#### Poor Use Cases:

1. **Unsupervised Tasks**
   - LDA requires class labels
   - Use PCA or other unsupervised methods instead

2. **Non-linear Class Boundaries**
   - LDA assumes linear separability
   - Consider Kernel LDA or non-linear methods

3. **Highly Imbalanced Classes**
   - Class with few samples may dominate scatter matrices
   - Consider resampling or weighted LDA

4. **Very High-dimensional Data (n_features >> n_samples)**
   - Scatter matrices may be singular
   - Use PCA first to reduce dimensions

### Assumptions of LDA

1. **Multivariate Normal Distribution**
   - Features should be approximately normally distributed within each class
   - LDA is robust to mild violations

2. **Equal Covariance Matrices**
   - All classes should have similar covariance (homoscedasticity)
   - If violated, consider Quadratic Discriminant Analysis (QDA)

3. **Independence of Features**
   - Correlated features can lead to multicollinearity
   - Regularization can help

4. **Sufficient Sample Size**
   - Need enough samples to estimate scatter matrices
   - Rule of thumb: at least 20 samples per feature per class

### Comparison Summary

| Scenario | Use LDA? | Alternative |
|----------|----------|-------------|
| Classification preprocessing | Yes | PCA if unsupervised |
| No class labels | No | PCA, t-SNE, UMAP |
| Non-linear boundaries | No | Kernel LDA, Neural Networks |
| Many classes (>10) | Maybe | Neural embeddings |
| Few samples per class | Maybe | Regularized LDA |
| Real-time prediction needed | Yes | Very fast transform |

In [ ]:
# Demonstrate assumption checking
def check_lda_assumptions(X, y, feature_names, class_names):
    """
    Check LDA assumptions visually.
    """
    n_classes = len(np.unique(y))
    
    # Select a few features for visualization
    features_to_check = [0, 6, 9, 12]  # alcohol, flavanoids, color_intensity, proline
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    
    for idx, feat_idx in enumerate(features_to_check):
        ax = axes[idx]
        
        # Plot distribution for each class
        for i, (color, name) in enumerate(zip(colors, class_names)):
            data = X[y == i, feat_idx]
            ax.hist(data, bins=15, alpha=0.5, color=color, 
                    label=f'{name} (n={len(data)})', density=True, edgecolor='black')
            
            # Overlay normal distribution
            from scipy import stats
            x_range = np.linspace(data.min() - 1, data.max() + 1, 100)
            pdf = stats.norm.pdf(x_range, data.mean(), data.std())
            ax.plot(x_range, pdf, color=color, linestyle='--', linewidth=2)
        
        ax.set_xlabel(feature_names[feat_idx], fontsize=11)
        ax.set_ylabel('Density', fontsize=11)
        ax.set_title(f'Distribution Check: {feature_names[feat_idx]}', fontsize=12)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
    
    plt.suptitle('Checking Normality Assumption for LDA\n(Dashed lines show fitted normal distributions)', 
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

check_lda_assumptions(X_train_scaled, y_train, feature_names, class_names)

## 7. Comparison with sklearn <a id='comparison'></a>

In [ ]:
# Compare our implementation with sklearn
print("Comparison: Our LDA vs sklearn LDA")
print("=" * 60)

# Our implementation
lda_ours = LDAScratch(n_components=2)
X_train_ours = lda_ours.fit_transform(X_train_scaled, y_train)
X_test_ours = lda_ours.transform(X_test_scaled)

# sklearn implementation
lda_sklearn = SklearnLDA(n_components=2)
X_train_sklearn = lda_sklearn.fit_transform(X_train_scaled, y_train)
X_test_sklearn = lda_sklearn.transform(X_test_scaled)

print("\nOur Implementation:")
print(f"  Explained variance ratio: {lda_ours.explained_variance_ratio_}")
print(f"  Sum of explained variance: {np.sum(lda_ours.explained_variance_ratio_):.4f}")

print("\nsklearn Implementation:")
print(f"  Explained variance ratio: {lda_sklearn.explained_variance_ratio_}")
print(f"  Sum of explained variance: {np.sum(lda_sklearn.explained_variance_ratio_):.4f}")

In [ ]:
# Visual comparison of projections
def compare_implementations(X_ours, X_sklearn, y, class_names):
    """
    Compare projections from our implementation vs sklearn.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    markers = ['o', 's', '^']
    
    # Our implementation
    for i, (color, marker, name) in enumerate(zip(colors, markers, class_names)):
        mask = y == i
        axes[0].scatter(X_ours[mask, 0], X_ours[mask, 1], 
                        c=color, marker=marker, s=80, alpha=0.7,
                        label=name, edgecolors='black', linewidth=0.5)
    axes[0].set_xlabel('LD1', fontsize=12)
    axes[0].set_ylabel('LD2', fontsize=12)
    axes[0].set_title('Our LDA Implementation', fontsize=14)
    axes[0].legend(title='Class')
    axes[0].grid(alpha=0.3)
    
    # sklearn implementation
    for i, (color, marker, name) in enumerate(zip(colors, markers, class_names)):
        mask = y == i
        axes[1].scatter(X_sklearn[mask, 0], X_sklearn[mask, 1], 
                        c=color, marker=marker, s=80, alpha=0.7,
                        label=name, edgecolors='black', linewidth=0.5)
    axes[1].set_xlabel('LD1', fontsize=12)
    axes[1].set_ylabel('LD2', fontsize=12)
    axes[1].set_title('sklearn LDA Implementation', fontsize=14)
    axes[1].legend(title='Class')
    axes[1].grid(alpha=0.3)
    
    plt.suptitle('Implementation Comparison: Our LDA vs sklearn LDA', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

compare_implementations(X_train_ours, X_train_sklearn, y_train, class_names)

In [ ]:
# Classification accuracy comparison
print("Classification Accuracy Comparison")
print("=" * 60)

# Train classifiers on both projections
clf_ours = LogisticRegression(max_iter=1000, random_state=42)
clf_sklearn = LogisticRegression(max_iter=1000, random_state=42)

clf_ours.fit(X_train_ours, y_train)
clf_sklearn.fit(X_train_sklearn, y_train)

print("\nOur LDA + Logistic Regression:")
print(f"  Train Accuracy: {clf_ours.score(X_train_ours, y_train):.4f}")
print(f"  Test Accuracy:  {clf_ours.score(X_test_ours, y_test):.4f}")

print("\nsklearn LDA + Logistic Regression:")
print(f"  Train Accuracy: {clf_sklearn.score(X_train_sklearn, y_train):.4f}")
print(f"  Test Accuracy:  {clf_sklearn.score(X_test_sklearn, y_test):.4f}")

In [ ]:
# Numerical comparison of components
def compare_components(lda_ours, lda_sklearn):
    """
    Compare the actual components (may differ by sign or scaling).
    """
    print("Component Comparison (may differ by sign/scale)")
    print("=" * 60)
    
    # Note: Components may have opposite signs but same direction
    for i in range(min(lda_ours.n_components, 2)):
        ours = lda_ours.components_[i]
        sklearn_comp = lda_sklearn.scalings_[:, i]
        
        # Normalize for comparison
        ours_norm = ours / np.linalg.norm(ours)
        sklearn_norm = sklearn_comp / np.linalg.norm(sklearn_comp)
        
        # Check if they point in same or opposite direction
        cos_sim = np.abs(np.dot(ours_norm, sklearn_norm))
        
        print(f"\nComponent {i+1}:")
        print(f"  Cosine similarity (absolute): {cos_sim:.6f}")
        print(f"  Directions are {'aligned' if cos_sim > 0.99 else 'different'}")

compare_components(lda_ours, lda_sklearn)

In [ ]:
# Performance comparison (timing)
import time

def benchmark_implementations(X, y, n_runs=10):
    """
    Benchmark execution time of both implementations.
    """
    # Our implementation
    times_ours = []
    for _ in range(n_runs):
        start = time.time()
        lda = LDAScratch(n_components=2)
        _ = lda.fit_transform(X, y)
        times_ours.append(time.time() - start)
    
    # sklearn implementation
    times_sklearn = []
    for _ in range(n_runs):
        start = time.time()
        lda = SklearnLDA(n_components=2)
        _ = lda.fit_transform(X, y)
        times_sklearn.append(time.time() - start)
    
    print("Execution Time Comparison")
    print("=" * 50)
    print(f"\nOur Implementation:")
    print(f"  Mean: {np.mean(times_ours)*1000:.3f} ms")
    print(f"  Std:  {np.std(times_ours)*1000:.3f} ms")
    
    print(f"\nsklearn Implementation:")
    print(f"  Mean: {np.mean(times_sklearn)*1000:.3f} ms")
    print(f"  Std:  {np.std(times_sklearn)*1000:.3f} ms")
    
    print(f"\nRatio (Ours/sklearn): {np.mean(times_ours)/np.mean(times_sklearn):.2f}x")
    
    return times_ours, times_sklearn

times_ours, times_sklearn = benchmark_implementations(X_train_scaled, y_train)

In [ ]:
# sklearn LDA additional features: classification
print("sklearn LDA Additional Features")
print("=" * 60)
print("\nsklearn's LDA can also perform classification directly:")

# sklearn LDA as classifier
lda_clf = SklearnLDA()
lda_clf.fit(X_train_scaled, y_train)

print(f"\nDirect classification accuracy:")
print(f"  Train: {lda_clf.score(X_train_scaled, y_train):.4f}")
print(f"  Test:  {lda_clf.score(X_test_scaled, y_test):.4f}")

print("\nNote: sklearn's LDA serves dual purpose:")
print("  1. Dimensionality reduction (transform)")
print("  2. Classification (predict)")

## Summary & Key Takeaways

### What We Learned:

1. **Theory**: LDA finds directions that maximize class separation using Fisher's criterion
2. **Implementation**: Computed scatter matrices and solved the generalized eigenvalue problem
3. **Constraints**: Maximum n_classes - 1 components (fundamental limit)
4. **Comparison**: LDA is supervised (uses labels), PCA is unsupervised (ignores labels)

### Key Insights:

- LDA reduced 13 features to 2 while maintaining excellent classification accuracy
- First component (LD1) captures most of the discriminative information
- LDA projections show much clearer class separation than PCA
- Our implementation produces results comparable to sklearn

### Practical Tips:

1. **Always scale features** before applying LDA
2. **Check assumptions** (normality, equal covariance) for best results
3. **Use regularization** when n_samples is small relative to n_features
4. **Combine with PCA** when n_features >> n_samples (PCA first, then LDA)

### Next Steps:

- Implement Quadratic Discriminant Analysis (QDA) for non-equal covariances
- Explore Kernel LDA for non-linear class boundaries
- Apply to real-world datasets (face recognition, medical diagnosis)
- Compare with other supervised dimensionality reduction methods